# Stage 5 — LLM Evidence Adjudication
### Arabic Herbal Medicine Validation Pipeline  |  Vertex AI / gemini-2.0-flash

**Inputs:** `claims_reference.xlsx` · `evidence_table.xlsx`

**Outputs:** `merged_claim_paper_pairs.xlsx` · `paper_level_scores.xlsx` · `final_claim_scores.xlsx`

> The LLM acts **only** as a constrained evidence reviewer.
> It scores what the abstract says. It never uses outside medical knowledge.

## Cell 1 — Upload Input Files

In [ ]:
from google.colab import files
print('Upload both files:')
print('  1. claims_reference.xlsx')
print('  2. evidence_table.xlsx')
uploaded = files.upload()


Upload both files:
  1. claims_reference.xlsx
  2. evidence_table.xlsx


Saving claims_reference_500.xlsx to claims_reference_500 (4).xlsx
Saving evidence_table_500.xlsx to evidence_table_500 (3).xlsx


## Cell 2 — Install Dependencies

In [ ]:
!pip install pandas openpyxl tqdm google-cloud-aiplatform -q


## Cell 3 — Authentication & Configuration

In [ ]:
!pip install --upgrade google-cloud-aiplatform -q
import pandas as pd
import json
import time
import os
import re
from tqdm import tqdm

# ═══════════════════════════════════════════════════════════════════════════
# AUTHENTICATION
# Each collaborator logs in with their own Gmail.
# Costs are charged to the project owner's Vertex AI credits.
# ═══════════════════════════════════════════════════════════════════════════
from google.colab import auth
auth.authenticate_user()
print('Authenticated!')

import vertexai
from vertexai.generative_models import GenerativeModel

PROJECT_ID = 'project-7f940e6f-ba1d-404b-948'
LOCATION   = 'global'
MODEL_ID   = 'gemini-3-flash-preview'

vertexai.init(project=PROJECT_ID, location=LOCATION)
model = GenerativeModel(MODEL_ID)
print(f'Vertex AI ready  |  project: {PROJECT_ID}  |  model: {MODEL_ID}')

# ── Sanity test ──────────────────────────────────────────────────────────────
test = model.generate_content('Reply with exactly the word: OK')
assert 'OK' in test.text.strip().upper(), f'Unexpected: {test.text}'
print(f'Sanity check passed!')

# ═══════════════════════════════════════════════════════════════════════════
# FILE PATHS
# ═══════════════════════════════════════════════════════════════════════════
CLAIMS_FILE     = 'claims_reference_500 (3).xlsx'
EVIDENCE_FILE   = 'evidence_table_500 (2).xlsx'
MERGED_FILE     = 'merged_claim_paper_pairs.xlsx'
PAPER_FILE      = 'paper_level_scores.xlsx'
FINAL_FILE      = 'final_claim_scores.xlsx'
CHECKPOINT_FILE = 'scoring_checkpoint.json'

# ═══════════════════════════════════════════════════════════════════════════
# SCORING CONFIG
# ═══════════════════════════════════════════════════════════════════════════
MAX_RETRIES      = 3     # retries per failed API call
RETRY_SLEEP      = 5     # seconds between retries
RATE_LIMIT_SLEEP = 30    # seconds to sleep on quota error
CHECKPOINT_EVERY = 50    # save checkpoint every N rows
SLEEP_BETWEEN    = 0.5   # seconds between API calls


Authenticated!
Vertex AI ready  |  project: project-7f940e6f-ba1d-404b-948  |  model: gemini-3-flash-preview


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Sanity check passed!


## Cell 4 — Load & Validate Input Files

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOAD FILES
# ═══════════════════════════════════════════════════════════════════════════
claims_df   = pd.read_excel(CLAIMS_FILE)
evidence_df = pd.read_excel(EVIDENCE_FILE)

# ── Fix 1: Fill null status values ───────────────────────────────────────────
# Papers that were successfully retrieved have null status (PubMed doesn't
# return a status field for found papers). Only 'No Evidence Found' rows
# have an explicit status. Fill nulls with 'found' for clarity.
evidence_df['status'] = evidence_df['status'].fillna('found')

# ── Fix 2: Fill null study_type values ───────────────────────────────────────
# 220 / 1322 papers have no study_type in PubMed metadata.
# Mark these explicitly so the scoring prompt handles them correctly.
evidence_df['study_type'] = evidence_df['study_type'].fillna('unknown')

print('=== claims_reference.xlsx ===')
print(f'Rows    : {len(claims_df)}')
print(f'Columns : {list(claims_df.columns)}')
print(f'Nulls   : {claims_df.isnull().sum().sum()} total')

print('\n=== evidence_table.xlsx ===')
print(f'Rows    : {len(evidence_df)}')
print(f'Columns : {list(evidence_df.columns)}')
print(f'\nstatus distribution:')
print(evidence_df['status'].value_counts().to_string())
print(f'\nstudy_type nulls remaining: {evidence_df["study_type"].isna().sum()}')

# ── Validate required columns ─────────────────────────────────────────────────
REQUIRED_CLAIMS   = {'claim_id','search_unit_id','entry_id','arabic_name',
                     'disease_or_condition_en','cluster_label',
                     'treatment_method_en','expected_effect_en'}
REQUIRED_EVIDENCE = {'search_unit_id','paper_title','abstract',
                     'pmid','year','study_type','source'}

missing_c = REQUIRED_CLAIMS   - set(claims_df.columns)
missing_e = REQUIRED_EVIDENCE - set(evidence_df.columns)

if missing_c:   print(f'\nWARNING: claims missing columns  : {missing_c}')
if missing_e:   print(f'WARNING: evidence missing columns: {missing_e}')
if not missing_c and not missing_e:
    print('\nAll required columns present — ready to merge.')

# ── search_unit_id overlap check ─────────────────────────────────────────────
claim_units    = set(claims_df['search_unit_id'].unique())
evidence_units = set(evidence_df['search_unit_id'].unique())
unmatched      = claim_units - evidence_units
print(f'\nUnique search_unit_ids in claims  : {len(claim_units)}')
print(f'Unique search_unit_ids in evidence: {len(evidence_units)}')
print(f'Unmatched (will get null papers)  : {len(unmatched)}')


=== claims_reference.xlsx ===
Rows    : 718
Columns : ['claim_id', 'search_unit_id', 'entry_id', 'arabic_name', 'disease_or_condition_en', 'cluster_label', 'treatment_method_en', 'expected_effect_en']
Nulls   : 0 total

=== evidence_table.xlsx ===
Rows    : 1322
Columns : ['paper_title', 'abstract', 'pmid', 'doi', 'year', 'journal_or_venue', 'study_type', 'source', 'search_unit_id', 'is_open_access', 'cited_by_count', 'status']

status distribution:
status
found                1284
No Evidence Found      38

study_type nulls remaining: 0

All required columns present — ready to merge.

Unique search_unit_ids in claims  : 500
Unique search_unit_ids in evidence: 500
Unmatched (will get null papers)  : 0


## Cell 5 — Merge & Build Claim-Paper Pairs

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MERGE — ONE ROW = ONE CLAIM-PAPER PAIR
# ═══════════════════════════════════════════════════════════════════════════
#
# Left join on search_unit_id.
# A single paper appears in multiple rows if multiple claims share one
# search unit — this is intentional, do NOT deduplicate.

merged = claims_df.merge(evidence_df, on='search_unit_id', how='left')
merged = merged.reset_index(drop=True)

# ── Build manuscript_claim field ──────────────────────────────────────────────
merged['manuscript_claim'] = (
    merged['arabic_name'].astype(str).str.strip()
    + ' treats '
    + merged['disease_or_condition_en'].astype(str).str.strip()
    + ' using '
    + merged['treatment_method_en'].astype(str).str.strip()
)

# ── Build unique pair_id for checkpointing ────────────────────────────────────
merged['pair_id'] = (
    merged['claim_id'].astype(str)
    + '_'
    + merged['pmid'].fillna('').astype(str)
    + '_'
    + merged.index.astype(str)
)

print(f'Claims rows           : {len(claims_df)}')
print(f'Evidence rows         : {len(evidence_df)}')
print(f'Merged pairs          : {len(merged)}')
print(f'Pairs with no abstract: {merged["abstract"].isna().sum()}')
print(f'\nSample manuscript_claim:')
for ex in merged['manuscript_claim'].dropna().head(3):
    print(f'  {ex}')

merged.to_excel(MERGED_FILE, index=False)
print(f'\nExported: {MERGED_FILE}')


Claims rows           : 718
Evidence rows         : 1322
Merged pairs          : 1911
Pairs with no abstract: 72

Sample manuscript_claim:
  الوسن treats Skin blemishes and scars using Topical application with honey
  الوسن treats Skin blemishes and scars using Topical application with honey
  الوسن treats Skin blemishes and scars using Topical application with honey

Exported: merged_claim_paper_pairs.xlsx


## Cell 6 — LLM Scoring Functions

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LLM PAPER-LEVEL SCORING ENGINE
# ═══════════════════════════════════════════════════════════════════════════

# ── Allowed field values for schema validation ────────────────────────────────
VALID_RELEVANCE = {'high', 'medium', 'low', 'irrelevant'}
VALID_SUPPORT   = {'supported', 'partially_supported', 'contradicted',
                   'insufficient_evidence', 'irrelevant'}
VALID_STRENGTH  = {'strong', 'moderate', 'weak', 'none'}
VALID_TX_MATCH  = {'exact', 'partial', 'unrelated', 'unclear'}
REQUIRED_FIELDS = {'relevance', 'support', 'evidence_strength',
                   'study_type_detected', 'treatment_match', 'confidence', 'reasoning'}

# ── Scoring prompt ────────────────────────────────────────────────────────────
# System instruction is embedded at the top — Vertex AI does not use
# a separate system message role.
SCORING_PROMPT_TEMPLATE = (
    'You are a constrained biomedical evidence reviewer in a research pipeline.\n'
    'Your ONLY job: evaluate whether a retrieved scientific abstract supports\n'
    'a historical Arabic herbal medicine claim.\n\n'

    'ABSOLUTE RULES — follow ALL of these without exception:\n'
    '1. Use ONLY information present in the abstract. Never use outside knowledge.\n'
    '2. Never hallucinate studies, compounds, or effects not in the abstract.\n'
    '3. Absence of evidence is NOT contradiction. If the abstract does not address\n'
    '   the claim, mark support as insufficient_evidence — NOT contradicted.\n'
    '4. Animal or in-vitro studies are weaker than human clinical trials.\n'
    '   Reflect this in evidence_strength: RCT/clinical = strong or moderate,\n'
    '   animal/in-vitro = weak, no data = none.\n'
    '5. Evaluate treatment_match SEPARATELY from disease support. The herb may\n'
    '   treat the disease but via a different method — that is partial, not exact.\n'
    '6. Empty or missing abstract → insufficient_evidence + irrelevant.\n'
    '7. Completely unrelated paper → irrelevant across all fields.\n'
    '8. If study_type_metadata is "unknown", infer the study type yourself\n'
    '   from the abstract content (e.g. RCT, in-vitro, review, case report).\n\n'

    'MANUSCRIPT CLAIM:\n{manuscript_claim}\n\n'
    'EXPECTED EFFECT:\n{expected_effect}\n\n'
    'DISEASE CLUSTER:\n{cluster_label}\n\n'
    'PAPER TITLE:\n{paper_title}\n\n'
    'ABSTRACT:\n{abstract}\n\n'
    'STUDY TYPE (from metadata): {study_type}\n'
    'YEAR: {year}\n\n'
    '---\n\n'
    'Return ONLY a valid JSON object. No markdown. No explanation. No preamble.\n'
    'Exact required format:\n'
    '{{\n'
    '  "relevance": "high | medium | low | irrelevant",\n'
    '  "support": "supported | partially_supported | contradicted | insufficient_evidence | irrelevant",\n'
    '  "evidence_strength": "strong | moderate | weak | none",\n'
    '  "study_type_detected": "inferred study type from abstract content",\n'
    '  "treatment_match": "exact | partial | unrelated | unclear",\n'
    '  "confidence": 0.0,\n'
    '  "reasoning": "1-3 sentences based strictly on abstract content"\n'
    '}}\n'
)


def build_prompt(row):
    """Build the scoring prompt for one claim-paper pair."""
    abstract = str(row.get('abstract', '') or '').strip()
    if not abstract or abstract.lower() in ('nan', 'none', ''):
        abstract = '[No abstract available]'
    return SCORING_PROMPT_TEMPLATE.format(
        manuscript_claim = str(row.get('manuscript_claim', '')).strip(),
        expected_effect  = str(row.get('expected_effect_en', '')).strip(),
        cluster_label    = str(row.get('cluster_label', '')).strip(),
        paper_title      = str(row.get('paper_title', '[No title]')).strip(),
        abstract         = abstract[:3000],
        study_type       = str(row.get('study_type', 'unknown')).strip(),
        year             = str(row.get('year', 'unknown')).strip(),
    )


def strip_fences(text):
    """Remove markdown fences the LLM sometimes wraps output in."""
    text = text.strip()
    if text.startswith('```'):
        text = text[text.index('\n') + 1:] if '\n' in text else text[3:]
    if text.endswith('```'):
        text = text[:text.rfind('```')]
    return text.strip()


def validate_score(record):
    """Validate parsed JSON against required schema. Returns (is_valid, errors)."""
    errors = []
    if not isinstance(record, dict):
        return False, ['Not a dict']
    missing = REQUIRED_FIELDS - set(record.keys())
    if missing:
        errors.append(f'Missing fields: {missing}')
    if record.get('relevance') not in VALID_RELEVANCE:
        errors.append(f'Invalid relevance: {record.get("relevance")}')
    if record.get('support') not in VALID_SUPPORT:
        errors.append(f'Invalid support: {record.get("support")}')
    if record.get('evidence_strength') not in VALID_STRENGTH:
        errors.append(f'Invalid evidence_strength: {record.get("evidence_strength")}')
    if record.get('treatment_match') not in VALID_TX_MATCH:
        errors.append(f'Invalid treatment_match: {record.get("treatment_match")}')
    try:
        conf = float(record.get('confidence', -1))
        if not (0.0 <= conf <= 1.0):
            errors.append(f'confidence out of range: {conf}')
    except (TypeError, ValueError):
        errors.append(f'confidence not a float: {record.get("confidence")}')
    return len(errors) == 0, errors


def _error_record(reason):
    """Safe fallback record when scoring fails completely."""
    return {
        'relevance': 'irrelevant', 'support': 'insufficient_evidence',
        'evidence_strength': 'none', 'study_type_detected': 'unknown',
        'treatment_match': 'unclear', 'confidence': 0.0,
        'reasoning': f'Scoring failed: {reason}',
        'scoring_error': reason
    }


def score_pair(row):
    """
    Score one claim-paper pair using Vertex AI gemini-2.0-flash.

    Pre-flight checks (no API call made):
      - Retracted papers → auto-flagged as insufficient_evidence
      - Missing abstract  → auto-flagged as insufficient_evidence

    Returns a dict with all scoring fields.
    """
    # ── Pre-flight check 1: Retracted papers ─────────────────────────────────
    # Retracted publications cannot be used as evidence.
    # Auto-flag without making an API call.
    study_type_raw = str(row.get('study_type', '') or '').lower()
    if 'retract' in study_type_raw:
        return {
            'relevance': 'irrelevant', 'support': 'insufficient_evidence',
            'evidence_strength': 'none', 'study_type_detected': 'retracted_publication',
            'treatment_match': 'unclear', 'confidence': 0.0,
            'reasoning': 'This paper has been retracted and cannot be used as evidence.',
            'scoring_error': 'retracted_publication'
        }

    # ── Pre-flight check 2: Missing abstract ─────────────────────────────────
    abstract = str(row.get('abstract', '') or '').strip()
    if not abstract or abstract.lower() in ('nan', 'none', ''):
        return {
            'relevance': 'irrelevant', 'support': 'insufficient_evidence',
            'evidence_strength': 'none', 'study_type_detected': 'unknown',
            'treatment_match': 'unclear', 'confidence': 0.0,
            'reasoning': 'No abstract available for this paper.',
            'scoring_error': 'no_abstract'
        }

    # ── LLM scoring ───────────────────────────────────────────────────────────
    prompt = build_prompt(row)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = model.generate_content(prompt)
            raw_text = response.text

        except Exception as e:
            err = str(e)
            if '429' in err or 'quota' in err.lower():
                print(f'  Rate limit — sleeping {RATE_LIMIT_SLEEP}s...')
                time.sleep(RATE_LIMIT_SLEEP)
            elif '403' in err or 'PERMISSION_DENIED' in err:
                print('  AUTH ERROR — check IAM permissions.')
                return _error_record('auth_error')
            elif '404' in err:
                print('  404 — Vertex AI API not enabled or wrong model ID.')
                return _error_record('model_not_found')
            else:
                print(f'  API error (attempt {attempt}): {e}')
                time.sleep(RETRY_SLEEP)
            continue

        # ── Parse JSON ────────────────────────────────────────────────────────
        cleaned = strip_fences(raw_text)
        try:
            record = json.loads(cleaned)
        except json.JSONDecodeError:
            match = re.search(r'\{.*\}', cleaned, re.DOTALL)
            if match:
                try:
                    record = json.loads(match.group())
                except json.JSONDecodeError:
                    print(f'  JSON parse failed (attempt {attempt}): {cleaned[:150]}')
                    time.sleep(RETRY_SLEEP)
                    continue
            else:
                print(f'  No JSON found (attempt {attempt}): {cleaned[:150]}')
                time.sleep(RETRY_SLEEP)
                continue

        # ── Validate schema ───────────────────────────────────────────────────
        is_valid, errors = validate_score(record)
        if not is_valid:
            if attempt < MAX_RETRIES:
                print(f'  Schema invalid (attempt {attempt}): {errors}')
                time.sleep(RETRY_SLEEP)
                continue
            else:
                record['scoring_error'] = f'schema_warnings: {errors}'
        else:
            record['scoring_error'] = None

        record['confidence'] = float(record.get('confidence', 0.0))
        return record

    return _error_record('max_retries_exceeded')


print('All scoring functions defined.')
print(f'Pre-flight checks: retracted papers + missing abstracts skipped automatically.')
print(f'Required output fields: {sorted(REQUIRED_FIELDS)}')


All scoring functions defined.
Pre-flight checks: retracted papers + missing abstracts skipped automatically.
Required output fields: ['confidence', 'evidence_strength', 'reasoning', 'relevance', 'study_type_detected', 'support', 'treatment_match']


## Cell 7 — Run Paper-Level Scoring

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PAPER-LEVEL SCORING WITH CHECKPOINTING
# ═══════════════════════════════════════════════════════════════════════════
#
# Saves progress to scoring_checkpoint.json every CHECKPOINT_EVERY rows.
# Re-running this cell skips already-scored pair_ids automatically.
# Safe to restart Colab runtime and re-run from here.

# ── Load checkpoint ───────────────────────────────────────────────────────────
checkpoint = {}
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
        checkpoint = json.load(f)
    print(f'Checkpoint loaded: {len(checkpoint)} pairs already scored.')
else:
    print('No checkpoint — starting fresh.')

remaining_df = merged[~merged['pair_id'].isin(checkpoint.keys())].copy()

print(f'Total pairs     : {len(merged)}')
print(f'Already scored  : {len(checkpoint)}')
print(f'Remaining       : {len(remaining_df)}')

# ── Pre-run summary ───────────────────────────────────────────────────────────
no_abstract_count = remaining_df['abstract'].isna().sum()
retracted_count   = remaining_df['study_type'].str.lower().str.contains('retract', na=False).sum()
will_call_api     = len(remaining_df) - no_abstract_count - retracted_count
print(f'\nPre-flight skips (no API call):')
print(f'  No abstract      : {no_abstract_count}')
print(f'  Retracted papers : {retracted_count}')
print(f'  Will call API    : {will_call_api}')
print(f'  Est. time @ 0.5s : ~{will_call_api * 0.5 / 60:.1f} minutes')

# ── Scoring loop ──────────────────────────────────────────────────────────────
scored_this_run = 0
errors_this_run = 0

for idx, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc='Scoring pairs'):
    result = score_pair(row)
    checkpoint[row['pair_id']] = result
    scored_this_run += 1

    if result.get('scoring_error'):
        errors_this_run += 1

    if scored_this_run % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)
        print(f'  Checkpoint: {len(checkpoint)} total scored.')

    time.sleep(SLEEP_BETWEEN)

# Final save
with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
    json.dump(checkpoint, f, ensure_ascii=False, indent=2)

print(f'\nScoring complete.')
print(f'Scored this run    : {scored_this_run}')
print(f'Errors this run    : {errors_this_run}')
print(f'Total in checkpoint: {len(checkpoint)}')


Checkpoint loaded: 800 pairs already scored.
Total pairs     : 1911
Already scored  : 800
Remaining       : 1111

Pre-flight skips (no API call):
  No abstract      : 41
  Retracted papers : 1
  Will call API    : 1069
  Est. time @ 0.5s : ~8.9 minutes


Scoring pairs:   4%|▍         | 49/1111 [12:05<3:54:37, 13.26s/it]

  Checkpoint: 850 total scored.


Scoring pairs:   5%|▌         | 61/1111 [15:31<4:27:19, 15.28s/it]


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 8 — Build & Export Paper-Level Scores

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BUILD paper_level_scores.xlsx
# ═══════════════════════════════════════════════════════════════════════════

score_rows = []
missing_from_checkpoint = 0

for idx, row in merged.iterrows():
    score = checkpoint.get(row['pair_id'])
    if score is None:
        score = _error_record('not_in_checkpoint')
        missing_from_checkpoint += 1
    combined = row.to_dict()
    combined.update({
        'llm_relevance'          : score.get('relevance'),
        'llm_support'            : score.get('support'),
        'llm_evidence_strength'  : score.get('evidence_strength'),
        'llm_study_type_detected': score.get('study_type_detected'),
        'llm_treatment_match'    : score.get('treatment_match'),
        'llm_confidence'         : score.get('confidence'),
        'llm_reasoning'          : score.get('reasoning'),
        'llm_scoring_error'      : score.get('scoring_error'),
    })
    score_rows.append(combined)

paper_scores_df = pd.DataFrame(score_rows)

if missing_from_checkpoint > 0:
    print(f'WARNING: {missing_from_checkpoint} pairs missing from checkpoint — re-run Cell 7.')

print(f'paper_level_scores rows  : {len(paper_scores_df)}')
print(f'Null llm_support         : {paper_scores_df["llm_support"].isna().sum()}')
print(f'\nSupport distribution:')
print(paper_scores_df['llm_support'].value_counts().to_string())
print(f'\nEvidence strength:')
print(paper_scores_df['llm_evidence_strength'].value_counts().to_string())
print(f'\nRelevance:')
print(paper_scores_df['llm_relevance'].value_counts().to_string())
print(f'\nScoring errors: {paper_scores_df["llm_scoring_error"].notna().sum()}')
print(f'  retracted   : {(paper_scores_df["llm_scoring_error"] == "retracted_publication").sum()}')
print(f'  no_abstract : {(paper_scores_df["llm_scoring_error"] == "no_abstract").sum()}')
print(f'  api_failures: {paper_scores_df["llm_scoring_error"].str.contains("retries|auth|model", na=False).sum()}')

paper_scores_df.to_excel(PAPER_FILE, index=False)
print(f'\nExported: {PAPER_FILE}')


paper_level_scores rows  : 1911
Null llm_support         : 0

Support distribution:
llm_support
insufficient_evidence    1547
irrelevant                260
partially_supported        78
supported                  16
contradicted               10

Evidence strength:
llm_evidence_strength
none        1555
weak         316
moderate      22
strong        18

Relevance:
llm_relevance
irrelevant    1449
low            228
medium         120
high           114

Scoring errors: 1092
  retracted   : 3
  no_abstract : 38
  api_failures: 0

Exported: paper_level_scores.xlsx


## Cell 9 — Claim-Level Aggregation

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CLAIM-LEVEL AGGREGATION
# ═══════════════════════════════════════════════════════════════════════════
#
# Aggregate all paper scores per claim_id into a single final verdict.
#
# Evidence strength ranking: strong > moderate > weak > none
#
# Final verdict rules:
#   supported            → supporting papers present, contradictions = 0
#   partially_supported  → mixed evidence (support + gaps or partial support)
#   contradicted         → contradicting papers >= supporting papers
#   insufficient_evidence→ no relevant papers at all

STRENGTH_RANK = {'strong': 4, 'moderate': 3, 'weak': 2, 'none': 1}


def get_strongest_evidence(strengths):
    """Return the highest evidence strength label from a list."""
    best = 0
    best_label = 'none'
    for s in strengths:
        rank = STRENGTH_RANK.get(str(s).lower(), 0)
        if rank > best:
            best = rank
            best_label = s
    return best_label


def derive_final_verdict(supporting, contradicting, total_relevant):
    """
    Derive final verdict from paper counts.
      supporting   = supported + partially_supported
      contradicting= contradicted
      total_relevant = supporting + contradicting
    """
    if total_relevant == 0:
        return 'insufficient_evidence'
    if contradicting > 0 and contradicting >= supporting:
        return 'contradicted'
    if supporting >= 2 and contradicting == 0:
        return 'supported'
    if supporting >= 1:
        return 'partially_supported'
    return 'insufficient_evidence'


def build_aggregation_reasoning(group):
    """Build a concise, traceable reasoning string from paper counts."""
    total   = len(group)
    sup     = int((group['llm_support'] == 'supported').sum())
    partial = int((group['llm_support'] == 'partially_supported').sum())
    contra  = int((group['llm_support'] == 'contradicted').sum())
    insuf   = int((group['llm_support'] == 'insufficient_evidence').sum())
    irrel   = int((group['llm_support'] == 'irrelevant').sum())
    strong  = int((group['llm_evidence_strength'] == 'strong').sum())
    rcts    = int(group['llm_study_type_detected'].str.lower()
                 .str.contains('rct|randomized|clinical trial', na=False).sum())
    retracted = int((group['llm_scoring_error'] == 'retracted_publication').sum())

    parts = [f'{total} papers reviewed']
    if sup:      parts.append(f'{sup} supporting')
    if partial:  parts.append(f'{partial} partially supporting')
    if contra:   parts.append(f'{contra} contradicting')
    if insuf:    parts.append(f'{insuf} insufficient evidence')
    if irrel:    parts.append(f'{irrel} irrelevant')
    if strong:   parts.append(f'{strong} strong-evidence papers')
    if rcts:     parts.append(f'{rcts} RCTs/clinical trials')
    if retracted:parts.append(f'{retracted} retracted (excluded)')
    return '. '.join(parts) + '.'


# ── Aggregate ─────────────────────────────────────────────────────────────────
aggregated_rows = []

for claim_id, group in paper_scores_df.groupby('claim_id'):
    # Exclude retracted papers from verdict calculation
    scoreable = group[group['llm_scoring_error'] != 'retracted_publication']

    supporting    = int(scoreable['llm_support'].isin(['supported','partially_supported']).sum())
    contradicting = int((scoreable['llm_support'] == 'contradicted').sum())
    irrelevant    = int(scoreable['llm_support'].isin(['irrelevant','insufficient_evidence']).sum())
    total_relevant= supporting + contradicting

    strengths   = scoreable['llm_evidence_strength'].dropna().tolist()
    avg_conf    = round(scoreable['llm_confidence'].dropna().astype(float).mean(), 3)
    strongest   = get_strongest_evidence(strengths)
    verdict     = derive_final_verdict(supporting, contradicting, total_relevant)
    reasoning   = build_aggregation_reasoning(group)

    first = group.iloc[0]
    aggregated_rows.append({
        'claim_id'                  : claim_id,
        'manuscript_claim'          : first.get('manuscript_claim', ''),
        'arabic_name'               : first.get('arabic_name', ''),
        'disease_or_condition_en'   : first.get('disease_or_condition_en', ''),
        'cluster_label'             : first.get('cluster_label', ''),
        'treatment_method_en'       : first.get('treatment_method_en', ''),
        'expected_effect_en'        : first.get('expected_effect_en', ''),
        'total_papers'              : len(group),
        'supporting_papers'         : supporting,
        'contradicting_papers'      : contradicting,
        'irrelevant_papers'         : irrelevant,
        'strongest_evidence_level'  : strongest,
        'average_confidence'        : avg_conf,
        'final_verdict'             : verdict,
        'aggregation_reasoning'     : reasoning,
    })

final_scores_df = pd.DataFrame(aggregated_rows)

print(f'Claims aggregated: {len(final_scores_df)}')
print(f'\nFinal verdict distribution:')
for verdict, count in final_scores_df['final_verdict'].value_counts().items():
    pct = 100 * count / len(final_scores_df)
    print(f'  {verdict:<25} {count:5d}  ({pct:.1f}%)')
print(f'\nStrongest evidence distribution:')
print(final_scores_df['strongest_evidence_level'].value_counts().to_string())
print(f'\nMean confidence: {final_scores_df["average_confidence"].mean():.3f}')


Claims aggregated: 718

Final verdict distribution:
  insufficient_evidence       639  (89.0%)
  partially_supported          51  (7.1%)
  supported                    19  (2.6%)
  contradicted                  9  (1.3%)

Strongest evidence distribution:
strongest_evidence_level
none        513
weak        174
moderate     16
strong       15

Mean confidence: 0.391


## Cell 10 — Export All Outputs

In [ ]:
merged.to_excel(MERGED_FILE, index=False)
paper_scores_df.to_excel(PAPER_FILE, index=False)
final_scores_df.to_excel(FINAL_FILE, index=False)

print(f'Saved: {MERGED_FILE}   ({len(merged)} rows)')
print(f'Saved: {PAPER_FILE}  ({len(paper_scores_df)} rows)')
print(f'Saved: {FINAL_FILE}   ({len(final_scores_df)} rows)')
print(f'\n=== PIPELINE SUMMARY ===')
print(f'Total claims scored    : {len(final_scores_df)}')
print(f'Total pairs processed  : {len(paper_scores_df)}')
print(f'\nFinal verdict breakdown:')
for v, c in final_scores_df['final_verdict'].value_counts().items():
    print(f'  {v:<28} {c:5d}  ({100*c/len(final_scores_df):.1f}%)')
print(f'\nScoring errors total   : {paper_scores_df["llm_scoring_error"].notna().sum()}')


## Cell 11 — Download

In [ ]:
from google.colab import files
files.download(MERGED_FILE)
files.download(PAPER_FILE)
files.download(FINAL_FILE)
print('All three files downloaded.')
